# 2. Dunder (Magic) Methods

"Dunder" = **D**ouble **UNDER**score. Methods like `__init__`, `__str__`, `__add__` let you hook into Python's built-in operations — making your objects work with `print()`, `+`, `==`, `for`, `in`, `len()`, and more.

**☕ Java equivalent:** `toString()`, `equals()`, `hashCode()`, `compareTo()` — but Python goes **much further** with full operator overloading, which Java does not support.

### 2.1 String Representation: `__str__` vs `__repr__`

**☕ JAVA:** Only `toString()` — one method for all purposes.

**🐍 PYTHON:** Two methods with different roles:

| Method | Purpose | Called by | Java Equivalent |
|--------|---------|----------|----------------|
| `__str__` | User-friendly display | `print()`, `str()`, f-strings | `toString()` |
| `__repr__` | Developer/debug string | REPL, `repr()`, containers | No equivalent! |

In [ ]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"
    
    def __repr__(self) -> str:
        return f"Point({self.x}, {self.y})"

p1 = Point(3, 4)
print(p1.x)
print(p1.y)
print(p1) # __str__

print(repr(p1)) # __repr__

3
4
(3, 4)
Point(3, 4)


> 💡 **Tip:** If you only define ONE, define `__repr__`. Python will fall back to `__repr__` when `__str__` is missing, but not the other way around.

### 2.2 Equality & Hashing: `__eq__` and `__hash__`

**☕ JAVA:**
```java
@Override public boolean equals(Object o) { ... }
@Override public int hashCode() { return Objects.hash(x, y); }
```

**🐍 PYTHON:** Same concept, but:
- Return `NotImplemented` (not `False`) when types don't match — this lets Python try the **other** operand's `__eq__`
- If you define `__eq__`, Python **removes** `__hash__` automatically (objects become unhashable). You must redefine `__hash__` if you want to use them in sets/dicts.

In [8]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"
    
    def __repr__(self) -> str:
        return f"Point({self.x}, {self.y})"
    
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Point):
            return False
            # return NotImplemented
        return self.x == other.x and self.y == other.y
    
    def __hash__(self) -> int:
        return hash((self.x, self.y))

p1 = Point(3, 4)
p2 = Point(3, 4)
p3 = Point(1, 2)

print(f"p1 == p2: {p1 == p2}")
print(f"p1 == p3: {p1 == p3}")
print(f"p1 is p2: {p1 is p2}")

print(f"Id(p1): {id(p1)}")
print(f"Id(p2): {id(p2)}")
print(f"Id(p3): {id(p3)}")

uniques = {p1, p2, p3}
print(f"Unique points: {uniques}")

p1 == p2: True
p1 == p3: False
p1 is p2: False
Id(p1): 131489177580992
Id(p2): 131489177699440
Id(p3): 131489177042912
Unique points: {Point(1, 2), Point(3, 4)}


### 2.3 Arithmetic Operators

**☕ JAVA:** No operator overloading at all (except `+` for String concatenation).

**🐍 PYTHON:** Full operator overloading! This is one of Python's superpowers.

| Operator | Dunder | Example |
|----------|--------|--------|
| `a + b` | `__add__` | Vector addition |
| `a - b` | `__sub__` | Vector subtraction |
| `a * b` | `__mul__` | Scalar multiplication |
| `b * a` | `__rmul__` | Reflected multiplication |
| `a / b` | `__truediv__` | Division |
| `-a` | `__neg__` | Negation |

> ⚠️ **Gotcha:** `v * 3` calls `v.__mul__(3)`, but `3 * v` calls `int.__mul__(v)` which fails! You need `__rmul__` (reflected) to handle the reversed case.

In [ ]:
class Vector:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y
 
    def __repr__(self) -> str:
        return f"Vector({self.x}, {self.y})"
 
    def __add__(self, other: "Vector") -> "Vector":
        """Enables: v1 + v2"""
        return Vector(self.x + other.x, self.y + other.y)
 
    def __sub__(self, other: "Vector") -> "Vector":
        """Enables: v1 - v2"""
        return Vector(self.x - other.x, self.y - other.y)
 
    def __mul__(self, scalar: float) -> "Vector":
        """Enables: v * 3"""
        return Vector(self.x * scalar, self.y * scalar)
 
    def __rmul__(self, scalar: float) -> "Vector":
        """Enables: 3 * v (reflected — called when left operand can't handle it)."""
        return self.__mul__(scalar)
 
    def __neg__(self) -> "Vector":
        """Enables: -v"""
        return Vector(-self.x, -self.y)
 
v1 = Vector(3, 4)
v2 = Vector(1, 2)
 
print(f"v1 + v2 = {v1 + v2}")
print(f"v1 - v2 = {v1 - v2}")
print(f"v1 * 3  = {v1 * 3}")
print(f"3 * v1  = {3 * v1}")    # Works because of __rmul__!
print(f"-v1     = {-v1}")

### 2.4 Comparison Operators

**☕ JAVA:**
```java
public class Point implements Comparable<Point> {
    @Override public int compareTo(Point other) {
        return Double.compare(this.magnitude(), other.magnitude());
    }
}
```

**🐍 PYTHON:** Individual methods for each operator:

| Operator | Dunder |
|----------|--------|
| `<` | `__lt__` |
| `<=` | `__le__` |
| `>` | `__gt__` |
| `>=` | `__ge__` |

> 💡 **Tip:** Use `@functools.total_ordering` — define just `__eq__` + `__lt__` and Python generates the rest!

In [21]:
import math
from functools import total_ordering

@total_ordering
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y
    
    @property
    def magnitude(self) -> float:
        return math.sqrt(self.x ** 2 + self.y ** 2)
    
    def __lt__(self, other: "Point") -> bool:
        return self.magnitude < other.magnitude
    
    def __eq__(self, other: "Point"):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y
    
p1 = Point(1, 1)
p2 = Point(3, 4)

print(f"p1 < p2: {p1 < p2}")
print(f"p1 > p2: {p1 > p2}")
print(f"p1 <= p2: {p1 <= p2}")
print(f"p1 >= p2: {p1 >= p2}")

p1 < p2: True
p1 > p2: False
p1 <= p2: True
p1 >= p2: False


### 2.5 Container Protocol: `__len__`, `__getitem__`, `__contains__`, `__reversed__`

**☕ JAVA:**
```java
list.size();        // → Python: len(obj)
list.get(0);        // → Python: obj[0]
list.contains(x);   // → Python: x in obj
```

**🐍 PYTHON:** Implement these dunders to make your objects work with `len()`, `[]`, `in`, and `reversed()`.

In [28]:
class Deck:
    def __init__(self):
        suits = ["♠", "♥", "♦", "♣"]
        ranks = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
        self.cards = [f"{r}{s}" for s in suits for r in ranks]

    def __len__(self) -> int:
        return len(self.cards)
    
    def __getitem__(self, index):
        return self.cards[index]
    
    def __contains__(self, card: str) -> bool:
        return card in self.cards
    
    def __reversed__(self):
        return reversed(self.cards)

deck = Deck()
print(f"Cards in deck: {len(deck)}")
print(f"First card: {deck[0]}")
print(f"Last card: {deck[-1]}")
print(f"First 5 cards: {deck[0:5]}")
print(f"Has A♦: {"A♦" in deck}")
print(list(reversed(deck)))

Cards in deck: 52
First card: A♠
Last card: K♣
First 5 cards: ['A♠', '2♠', '3♠', '4♠', '5♠']
Has A♦: True
['K♣', 'Q♣', 'J♣', '10♣', '9♣', '8♣', '7♣', '6♣', '5♣', '4♣', '3♣', '2♣', 'A♣', 'K♦', 'Q♦', 'J♦', '10♦', '9♦', '8♦', '7♦', '6♦', '5♦', '4♦', '3♦', '2♦', 'A♦', 'K♥', 'Q♥', 'J♥', '10♥', '9♥', '8♥', '7♥', '6♥', '5♥', '4♥', '3♥', '2♥', 'A♥', 'K♠', 'Q♠', 'J♠', '10♠', '9♠', '8♠', '7♠', '6♠', '5♠', '4♠', '3♠', '2♠', 'A♠']


### 2.6 `__bool__` — Truthiness

**☕ JAVA:** No direct equivalent. You'd call `isEmpty()` or check `size() > 0`.

**🐍 PYTHON:** `__bool__` lets you use `if obj:` directly. If not defined, Python falls back to `__len__` (truthy if non-zero).

In [ ]:
my_list = []

if my_list:
    print(my_list)
else:
    print("The list is empty")

In [32]:
class MyList:
    def __init__(self, data: list):
        self.data = data
    
    def __bool__(self):
        return True if len(self.data) > 0 else False

my_list = MyList([10, 15])

if my_list:
    print("Bingo")

Bingo


In [31]:
class Wallet:
    def __init__(self, balance: float = 0):
        self.balance = balance

    def __bool__(self):
        return self.balance > 800_000

rich = Wallet(1_000_000)
poor = Wallet(0)

if poor:
    print("You are VIP member")
else:
    print("Goodbye")

Goodbye


### 2.7 `__call__` — Callable Objects

**☕ JAVA:** Closest equivalent: `Callable<T>` interface or lambda expressions.

**🐍 PYTHON:** `__call__` lets you use an object like a function: `obj()`. Great for stateful functions or configurable behaviors.

In [33]:
class Multiplier:
    def __init__(self, factor: float):
        self.factor = factor
    
    def __call__(self, value: float):
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(f"double(5): {double(5)}")
print(f"triple(5): {triple(5)}")


double(5): 10
triple(5): 15


### 2.8 Context Manager: `__enter__` and `__exit__`

**☕ JAVA:** `try-with-resources` requires `implements AutoCloseable` with a `close()` method.

**🐍 PYTHON:** `__enter__` and `__exit__` let you use the `with` statement. `__enter__` sets up the resource, `__exit__` cleans it up — **even if an exception occurs**.

In [5]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed time: {self.elapsed:.6f}s")
        return False
    
with Timer() as t:
    total = sum(range(1_000_000))
    print(f"Sum: {total}")


Sum: 499999500000
Elapsed time: 0.044980s


In [6]:
class DatabaseConnection:
    def __init__(self, db_name: str):
        self.db_name = db_name
        self.connected = False
    
    def __enter__(self):
        self.connected = True
        print(f"Connected to {self.db_name}")
        return self
    
    def __exit__(self, exc_type, exc, tb):
        self.connected = False
        print(f"Disconnected from {self.db_name}")
        if exc_type:
            print(f" -- Error occured: {exc}")
        return False
    
with DatabaseConnection("ai_for_devs_db") as db:
    print(f" Connected: {db.connected}")

print("-" * 70)
print(f"After with: {db.connected}")

Connected to ai_for_devs_db
 Connected: True
Disconnected from ai_for_devs_db
----------------------------------------------------------------------
After with: False


> 💡 **`__exit__` parameters:** `exc_type`, `exc_val`, `exc_tb` receive exception info if one occurred inside the `with` block. Return `True` to suppress the exception, `False` to propagate it.

---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Money` class with `amount` and `currency`. Implement `__add__` (only if same currency), `__str__`, and `__repr__`.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a `ShoppingCart` class that supports `len(cart)`, `cart[0]`, and `item in cart` using container dunders.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a `Counter` class with `__call__` that increments and returns a count each time it's called.

In [ ]:
# Exercise 3: Your code here


**Exercise 4 ⭐⭐⭐:** Create a `Matrix` class (2×2) with:
- `__init__(a, b, c, d)` for `[[a, b], [c, d]]`
- `__add__`: matrix addition
- `__mul__` and `__rmul__`: scalar multiplication (both `M * 3` and `3 * M`)
- `__repr__`: `"Matrix([[a, b], [c, d]])"`
- `__eq__`: compare by values
- `__bool__`: `False` if all elements are zero (zero matrix)
- Context manager: `__enter__` prints "Matrix operation started", `__exit__` prints "Matrix operation completed"

In [ ]:
# Exercise 4: Your code here


---

## 📝 Key Takeaways: Java → Python

| Dunder | Python Syntax | Java Equivalent |
|--------|--------------|----------------|
| `__str__` | `print(obj)`, `str(obj)` | `toString()` |
| `__repr__` | `repr(obj)`, shown in REPL | No equivalent |
| `__eq__` | `a == b` | `equals()` |
| `__hash__` | `hash(obj)`, sets/dicts | `hashCode()` |
| `__lt__` / `__le__` / `__gt__` / `__ge__` | `<` `<=` `>` `>=` | `compareTo()` |
| `__add__` / `__sub__` / `__mul__` | `+` `-` `*` | No operator overloading! |
| `__rmul__` | `3 * obj` (reflected) | No operator overloading! |
| `__len__` | `len(obj)` | `.size()` |
| `__getitem__` | `obj[i]` | `.get(i)` |
| `__contains__` | `x in obj` | `.contains(x)` |
| `__reversed__` | `reversed(obj)` | N/A |
| `__bool__` | `if obj:` | `.isEmpty()` check |
| `__call__` | `obj()` | `Callable<T>` interface |
| `__enter__` / `__exit__` | `with obj:` | `AutoCloseable` + `try-with-resources` |
| `@total_ordering` | Auto-generates `<=`, `>`, `>=` | N/A |